<a href="https://colab.research.google.com/github/MeouchyCharbel/Neural-Machine-Translation/blob/feature%2Fevaluation-improvements/03_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import torch
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    gpu_memory_gb = gpu_memory / (1024 ** 3)

    print("GPU memory:", round(gpu_memory_gb, 2), "GB")
else:
    print("No GPU detected. Check Colab runtime settings.")


Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:

%pip install -q transformers datasets accelerate evaluate sentencepiece sacremoses sacrebleu scikit-learn pandas


In [3]:

import csv
import glob
import hashlib
import urllib.request
import zipfile
import pandas as pd

DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip"
ZIP_PATH = "fra-eng.zip"
EXTRACT_DIR = "data_raw"

urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

with open(ZIP_PATH, "rb") as file:
    dataset_hash = hashlib.file_digest(file, "sha256").hexdigest()

with zipfile.ZipFile(ZIP_PATH) as zip_file:
    zip_file.extractall(EXTRACT_DIR)

text_file = glob.glob(
    f"{EXTRACT_DIR}/**/fra.txt",
    recursive=True
)[0]

pairs = pd.read_csv(
    text_file,
    sep="\t",
    header=None,
    usecols=[0, 1],
    names=["en", "fr"],
    quoting=csv.QUOTE_NONE,
    encoding="utf-8"
)

rows_before = len(pairs)

pairs = pairs.dropna(subset=["en", "fr"]).copy()

pairs["en"] = pairs["en"].str.strip()
pairs["fr"] = pairs["fr"].str.strip()

pairs["en_words"] = pairs["en"].str.split().str.len()

pairs = pairs[
    (pairs["en"] != "") &
    (pairs["fr"] != "")
]

pairs = pairs.drop_duplicates(subset=["en", "fr"])
pairs = pairs[pairs["en_words"] <= 15]
pairs = pairs.reset_index(drop=True)

print("Original sentence pairs:", rows_before)
print("Cleaned sentence pairs:", len(pairs))
print("Unique English sentences:", pairs["en"].nunique())
print("Dataset SHA-256:", dataset_hash)

display(pairs[["en", "fr"]].head(5))


Original sentence pairs: 167130
Cleaned sentence pairs: 166340
Unique English sentences: 116198
Dataset SHA-256: 036e7dc4e83d393db24720ca0c7b7ddcb4b6b6053960438a5c0e1d6d8f2528f5


,en,fr
0,Go.,Va !
1,Hi.,Salut !
2,Run!,Cours !
3,Run!,Courez !
4,Who?,Qui ?


In [4]:

import re
import unicodedata
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def loose_normalize(text):
    text = normalize_text(text)
    text = re.sub(r"[\W_]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

pairs["en_key"] = pairs["en"].apply(normalize_text)
pairs["fr_key"] = pairs["fr"].apply(normalize_text)
pairs["en_loose"] = pairs["en"].apply(loose_normalize)
pairs["fr_loose"] = pairs["fr"].apply(loose_normalize)

parent = list(range(len(pairs)))
size = [1] * len(pairs)

def find(index):
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a == root_b:
        return

    if size[root_a] < size[root_b]:
        root_a, root_b = root_b, root_a

    parent[root_b] = root_a
    size[root_a] += size[root_b]

seen = {}

for i, (english, french) in enumerate(
    zip(pairs["en_loose"], pairs["fr_loose"])
):
    for key in [("en", english), ("fr", french)]:
        if key in seen:
            union(i, seen[key])
        else:
            seen[key] = i

pairs["group_id"] = [
    find(i) for i in range(len(pairs))
]

first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, remaining_idx = next(
    first_split.split(
        pairs,
        groups=pairs["group_id"]
    )
)

train_pairs = pairs.iloc[train_idx].reset_index(drop=True)
remaining_pairs = pairs.iloc[remaining_idx].reset_index(drop=True)

second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=RANDOM_STATE
)

val_idx, test_idx = next(
    second_split.split(
        remaining_pairs,
        groups=remaining_pairs["group_id"]
    )
)

val_pairs = remaining_pairs.iloc[val_idx].reset_index(drop=True)
test_pairs = remaining_pairs.iloc[test_idx].reset_index(drop=True)

assert len(train_pairs) == 133017
assert len(val_pairs) == 16616
assert len(test_pairs) == 16707

splits = {
    "Train": train_pairs,
    "Validation": val_pairs,
    "Test": test_pairs
}

columns_to_check = [
    "en_key",
    "fr_key",
    "en_loose",
    "fr_loose",
    "group_id"
]

comparisons = [
    ("Train", "Validation"),
    ("Train", "Test"),
    ("Validation", "Test")
]

for left_name, right_name in comparisons:
    for column in columns_to_check:
        overlap = (
            set(splits[left_name][column])
            & set(splits[right_name][column])
        )

        assert len(overlap) == 0

assert sum(len(df) for df in splits.values()) == len(pairs)

print("DATASET SPLITTING VERIFIED")
print("Training:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Testing:", len(test_pairs))
print("Connected groups:", pairs["group_id"].nunique())
print("Largest group:", pairs["group_id"].value_counts().max())
print("All exact and normalized overlap checks passed.")


DATASET SPLITTING VERIFIED
Training: 133017
Validation: 16616
Testing: 16707
Connected groups: 108720
Largest group: 96
All exact and normalized overlap checks passed.


In [5]:

import hashlib
import json

SEED = 42
TRAIN_SAMPLE_SIZE = 20000
VALIDATION_SAMPLE_SIZE = 1000

training_subset = train_pairs.sample(
    n=min(TRAIN_SAMPLE_SIZE, len(train_pairs)),
    random_state=SEED
).reset_index(drop=True)

validation_subset = val_pairs.sample(
    n=min(VALIDATION_SAMPLE_SIZE, len(val_pairs)),
    random_state=SEED
).reset_index(drop=True)

train_groups = set(training_subset["group_id"])
validation_groups = set(validation_subset["group_id"])
test_groups = set(test_pairs["group_id"])

assert train_groups.isdisjoint(validation_groups)
assert train_groups.isdisjoint(test_groups)
assert validation_groups.isdisjoint(test_groups)

def dataset_fingerprint(data):
    records = sorted(
        zip(data["en"].tolist(), data["fr"].tolist())
    )

    content = json.dumps(
        records,
        ensure_ascii=False,
        separators=(",", ":")
    )

    return hashlib.sha256(
        content.encode("utf-8")
    ).hexdigest()

train_fingerprint = dataset_fingerprint(training_subset)
validation_fingerprint = dataset_fingerprint(validation_subset)

print("FINE-TUNING DATASET")
print("Training pairs:", len(training_subset))
print("Validation pairs:", len(validation_subset))
print("Random seed:", SEED)
print("Training fingerprint:", train_fingerprint)
print("Validation fingerprint:", validation_fingerprint)
print("Group overlap checks: Passed")

display(
    training_subset[["en", "fr"]].head(5)
)


FINE-TUNING DATASET
Training pairs: 20000
Validation pairs: 1000
Random seed: 42
Training fingerprint: 99887fd07f9b20eb72d50473b9608cd565f7ab5e615e3ed89384765bfb3a74b3
Validation fingerprint: 1e4e9761bcf231fbe6e0cf769a63d42a34ea8081da7556825ac0193120f77d13
Group overlap checks: Passed


,en,fr
0,"I just think you should be careful, that's all.",Je pense simplement que vous devriez être prud...
1,Who asked you to do that?,Qui t'a demandé de faire ça ?
2,I always thought Tom would outlive me.,J'ai toujours pensé que Tom me survivrait.
3,I like to eat apples.,J'aime manger des pommes.
4,Where would you want to live?,Où voudrais-tu vivre ?
